# Generate web search eval JSONL

Read prompt records from `data/search_eval.jsonl`, call the Responses API with the `web_search_preview` tool enabled, and write raw response payloads to `outputs/web_search_eval_raw_<timestamp>.jsonl`.

Required environment variables (load from `.env`):
- `AZURE_OPENAI_BASE_URL` (or `AZURE_OPENAI_API_BASE` / `AZURE_EXISTING_AIPROJECT_ENDPOINT`)
- `AZURE_OPENAI_API_KEY` (or `OPENAI_API_KEY`)
- `AZURE_OPENAI_MODEL` (or `AZURE_OPENAI_DEPLOYMENT`)

In [ ]:
# Load standard libraries, dotenv, and OpenAI client helpers for the eval run.
import json
import os
import time
from datetime import datetime
from pathlib import Path

from dotenv import load_dotenv
from openai import OpenAI, RateLimitError

load_dotenv(override=True)

In [ ]:
# Read the evaluation prompts and ensure each record has a stable identifier.
data_path = Path("../data/search_eval.jsonl")
records = [
    json.loads(line)
    for line in data_path.read_text(encoding="utf-8").splitlines()
    if line.strip()
]
for index, record in enumerate(records, start=1):
    record.setdefault("id", f"search_{index:03d}")

records[:3]

In [ ]:
# Resolve environment configuration and build a Responses API client.
base_url = (
    os.getenv("AZURE_OPENAI_BASE_URL")
    or os.getenv("AZURE_OPENAI_API_BASE")
    or os.getenv("AZURE_EXISTING_AIPROJECT_ENDPOINT")
)
api_key = os.getenv("AZURE_OPENAI_API_KEY") or os.getenv("OPENAI_API_KEY")
if not base_url or not api_key:
    raise ValueError("Set base URL and API key in .env.")

base_url = base_url.rstrip("/")
if not base_url.endswith("/openai/v1"):
    base_url = f"{base_url}/openai/v1"

model = (
    os.getenv("AZURE_OPENAI_MODEL")
    or os.getenv("AZURE_OPENAI_DEPLOYMENT")
    or "gpt-4.1-mini"
)
client = OpenAI(api_key=api_key, base_url=base_url)

In [ ]:
# Call the Responses API with web search enabled and capture raw payloads.
sleep_seconds = 2.0
responses: list[dict[str, object]] = []

for record in records:
    try:
        response = client.responses.create(
            model=model,
            tools=[{"type": "web_search_preview"}],
            input=record["query"],
        )
        responses.append(
            {
                "id": record.get("id"),
                "query": record.get("query"),
                "test_case_description": record.get("test_case_description"),
                "response": response.model_dump(),
            }
        )
    except RateLimitError as exc:
        responses.append(
            {
                "id": record.get("id"),
                "query": record.get("query"),
                "test_case_description": record.get("test_case_description"),
                "error": {"type": "RateLimitError", "message": str(exc)},
            }
        )
    if sleep_seconds:
        time.sleep(sleep_seconds)

len(responses)

In [ ]:
# Persist the raw response payloads as a JSONL log for evaluation.
output_dir = Path("../outputs")
output_dir.mkdir(parents=True, exist_ok=True)
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
output_path = output_dir / f"web_search_eval_raw_{timestamp}.jsonl"

payload = "\n".join(json.dumps(item, ensure_ascii=False) for item in responses) + "\n"
output_path.write_text(payload, encoding="utf-8")
output_path

In [ ]:
# Quick sanity check: how many responses succeeded or failed.
total = len(responses)
error_count = sum(1 for item in responses if "error" in item)
(total, error_count)